# House Price Prediction — Preprocessing & EDA

This notebook covers initial data exploration, feature engineering, and preprocessing of Zoopla property listings data for West Yorkshire, UK.

**Steps:**
1. Load and explore the raw dataset
2. Data type corrections
3. Feature engineering (borough mapping, property grouping)
4. Exploratory Data Analysis (univariate, bivariate)
5. Final preprocessing (drop irrelevant features, handle missing values)
6. Save cleaned dataset for modelling

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

print('All imports successful')

## 2. Load Dataset

Data scraped from Zoopla covering property listings in West Yorkshire (Dec–Jan).

In [ ]:
df = pd.read_csv('../data/house_prices_west_yorkshire.csv', sep='\t')
print(f'Dataset shape: {df.shape}')
df.head()

## 3. Initial Exploration

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Duplicates:', df.duplicated().any())
print('\nMissing values:')
print(df.isna().sum())

## 4. Data Type Corrections

`numBaths` and `numRecepts` are stored as floats but represent counts — converting to nullable integers. `isRetirementHome` and `isSharedOwnership` are boolean flags stored as numeric — converting to object for categorical treatment.

In [ ]:
df['numBaths'] = df['numBaths'].astype('Int64')
df['numRecepts'] = df['numRecepts'].astype('Int64')
df['isRetirementHome'] = df['isRetirementHome'].astype('object')
df['isSharedOwnership'] = df['isSharedOwnership'].astype('object')
df.info()

## 5. Feature Engineering

### 5a. Map Postcodes to Metropolitan Boroughs

The `outcode` column contains postcode prefixes (e.g. LS, BD, WF). We map these to the five West Yorkshire metropolitan boroughs to create a higher-level geographic feature.

In [ ]:
def map_outcode_to_metborough(outcode):
    if outcode.startswith('WF'):
        return 'Wakefield'
    elif outcode.startswith('HD'):
        return 'Kirklees'
    elif outcode.startswith('HX'):
        return 'Calderdale'
    elif outcode.startswith('BD'):
        return 'Bradford'
    elif outcode.startswith('LS'):
        return 'Leeds'
    else:
        return None

df['MetBoroughs'] = df['outcode'].apply(map_outcode_to_metborough)
print('Borough distribution:')
print(df['MetBoroughs'].value_counts())

### 5b. Group Property Types

The raw `propertyType` column has many granular categories. We group these into broader categories (House, Flat, Bungalow, etc.) to reduce dimensionality.

In [ ]:
property_type_mapping = {
    'detached': 'House', 'semi_detached': 'House', 'end_terrace': 'House',
    'terraced': 'House', 'town_house': 'House', 'mews': 'House', 'link_detached': 'House',
    'flat': 'Flat', 'block_of_flats': 'Flat', 'maisonette': 'Flat',
    'bungalow': 'Bungalow', 'detached_bungalow': 'Bungalow',
    'semi_detached_bungalow': 'Bungalow', 'terraced_bungalow': 'Bungalow',
    'park_home': 'Mobile Home', 'lodge': 'Mobile Home', 'chalet': 'Mobile Home',
    'cottage': 'Country Home', 'country_house': 'Country Home', 'farmhouse': 'Country Home',
    'barn_conversion': 'Converted Property',
    'land': 'Non-Residential', 'parking': 'Non-Residential', 'retail': 'Non-Residential',
}

df['propertyGroup'] = df['propertyType'].map(property_type_mapping).fillna('Unknown')
print('Property group distribution:')
print(df['propertyGroup'].value_counts())

### 5c. Select Relevant Features

In [ ]:
df_new = df[[
    'filename', 'rooms', 'isRetirementHome', 'isSharedOwnership', 'listingCondition',
    'location', 'numBaths', 'numRecepts', 'outcode', 'postTownName', 'MetBoroughs',
    'propertyType', 'propertyGroup', 'tenure', 'ditsnace_to_school', 'ditsnace_to_train',
    'text_description', 'features', 'price'
]]
print(f'Selected features shape: {df_new.shape}')

## 6. Exploratory Data Analysis

### 6a. Numerical Features

In [ ]:
num_data = df_new.select_dtypes(exclude='object')
num_column = num_data.columns
print('Numerical columns:', list(num_column))

### 6b. Categorical Features

In [ ]:
cat_columns = ['MetBoroughs', 'isRetirementHome', 'isSharedOwnership', 'listingCondition', 'propertyGroup', 'tenure']
cat_data = df_new[cat_columns]

## 7. Univariate Analysis

Examining the distribution of individual features.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

sns.countplot(x='rooms', data=df_new, ax=axes[0, 0])
axes[0, 0].set_title('Count of Houses by Number of Rooms')

sns.countplot(x='numBaths', data=df_new, ax=axes[0, 1])
axes[0, 1].set_title('Count of Bathrooms per House')

sns.countplot(x='numRecepts', data=df_new, ax=axes[0, 2])
axes[0, 2].set_title('Distribution of Number of Receptions')

sns.histplot(df_new['ditsnace_to_school'], bins=20, kde=True, ax=axes[1, 0])
axes[1, 0].set_title('Distribution of Distance to Nearest School')

sns.histplot(df_new['ditsnace_to_train'], bins=20, kde=True, ax=axes[1, 1])
axes[1, 1].set_title('Distance to Train Station')

sns.histplot(df_new['price'], bins=50, kde=True, ax=axes[1, 2])
axes[1, 2].set_title('Distribution of House Prices')

plt.tight_layout()
plt.show()

### Log-Transformed Price Distribution

House prices are typically right-skewed. A log transformation reveals a more normal distribution which can improve model performance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_new['price'], bins=50, kde=True, ax=axes[0])
axes[0].set_title('House Price Distribution (Original)')
sns.histplot(np.log1p(df_new['price']), bins=50, kde=True, ax=axes[1])
axes[1].set_title('House Price Distribution (Log-Transformed)')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots for all numerical features
melted = df_new[num_column].melt(var_name='Feature', value_name='Value')
plt.figure(figsize=(12, 6))
sns.boxplot(x='Feature', y='Value', data=melted)
plt.title('Boxplots of Numerical Features')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution plots for categorical features
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_columns):
    sns.countplot(data=cat_data, x=col, order=cat_data[col].value_counts().index, ax=axes[i])
    axes[i].set_title(f'Distribution of {col}')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 8. Bivariate Analysis

### 8a. Numerical vs Numerical

In [ ]:
sns.pairplot(df_new[num_column])
plt.show()

In [ ]:
num_data_corr = df_new.select_dtypes(include=['int64', 'float64', 'Int64'])
plt.figure(figsize=(10, 7))
sns.heatmap(num_data_corr.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

### 8b. Categorical vs Numerical (Price Analysis)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(data=df_new, x='tenure', y='price', ax=axes[0])
axes[0].set_title('Average Price by Tenure')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(data=df_new, x='propertyGroup', y='price', estimator='mean', ax=axes[1])
axes[1].set_title('Average Price by Property Type')
axes[1].tick_params(axis='x', rotation=45)

sns.barplot(data=df_new, x='listingCondition', y='price', ax=axes[2])
axes[2].set_title('Average Price by Listing Condition')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
df_new.groupby('MetBoroughs')['price'].mean().sort_values().plot(kind='barh', figsize=(8, 5))
plt.title('Average Price by Borough')
plt.xlabel('Mean Price (£)')
plt.tight_layout()
plt.show()

### 8c. Categorical vs Categorical

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

pd.crosstab(df_new['MetBoroughs'], df_new['propertyGroup']).plot(
    kind='bar', stacked=True, figsize=(8, 5), ax=axes[0])
axes[0].set_title('Property Types by Borough')
axes[0].tick_params(axis='x', rotation=45)

pd.crosstab(df_new['propertyGroup'], df_new['listingCondition']).plot(
    kind='bar', stacked=True, ax=axes[1])
axes[1].set_title('Property Types by Listing Condition')
axes[1].tick_params(axis='x', rotation=45)

pd.crosstab(df_new['propertyGroup'], df_new['tenure']).plot(
    kind='bar', stacked=True, ax=axes[2])
axes[2].set_title('Property Types by Tenure')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 9. Final Preprocessing

### 9a. Drop Irrelevant Features

Removing columns that are not useful for modelling: free text fields (`text_description`, `features`), identifier fields (`filename`), redundant location fields (`location`, `outcode`, `postTownName`), and the granular `propertyType` (replaced by `propertyGroup`).

In [ ]:
df_new = df_new.drop(['filename', 'location', 'outcode', 'postTownName', 'propertyType',
                      'text_description', 'features'], axis=1)
print(f'Shape after dropping irrelevant features: {df_new.shape}')
df_new.head()

### 9b. Handle Missing Values

- `MetBoroughs`: rows where the outcode couldn't be mapped are dropped (small proportion)
- `numBaths`, `numRecepts`: filled with median values
- `tenure`: filled proportionally based on existing category distribution to avoid introducing bias

In [ ]:
# Drop rows where borough could not be mapped
df_new.dropna(subset=['MetBoroughs'], inplace=True)

# Fill numBaths and numRecepts with median
df_new['numBaths'].fillna(df_new['numBaths'].median(), inplace=True)
df_new['numRecepts'].fillna(df_new['numRecepts'].median(), inplace=True)

# Fill tenure proportionally
tenure_proportions = df_new['tenure'].value_counts(normalize=True)
null_count = df_new['tenure'].isnull().sum()
null_replacements = np.random.choice(tenure_proportions.index, size=null_count, p=tenure_proportions.values)
df_new.loc[df_new['tenure'].isnull(), 'tenure'] = null_replacements

print('Missing values after handling:')
print(df_new.isna().sum())

## 10. Save Preprocessed Dataset

In [ ]:
df_new.to_csv('../data/preprocessed_zoopla.csv', index=False)
print(f'Saved preprocessed dataset: {df_new.shape[0]} rows, {df_new.shape[1]} columns')
print('Features:', list(df_new.columns))